# Track B — 추론 → submission.csv (Kaggle T4)

선행: `snuai_code` Dataset, 대회 데이터, 학습 노트북이 만든 `qwen25vl3b_merged` Dataset attach.

permutation-TTA 생성(predict) → 투표 집계(aggregate) → 검증(submission). 예상 test 819 × TTA4 ≈ 1~2h.

In [ ]:
import os, sys, glob, shutil
CODE = glob.glob('/kaggle/input/*/src')[0].rsplit('/src', 1)[0]
sys.path.insert(0, CODE)

hits = sorted(glob.glob('/kaggle/input/*/train.csv')
              + glob.glob('/kaggle/input/*/*/train.csv')
              + glob.glob('/kaggle/input/*/*/*/train.csv'))
assert hits, '대회 데이터가 attach되지 않았습니다'
DATA_ROOT = os.path.dirname(hits[0])
yaml_text = '\n'.join([
    f'data_dir: {DATA_ROOT}',
    f'train_csv: {DATA_ROOT}/train.csv',
    f'test_csv: {DATA_ROOT}/test.csv',
    f'sample_submission: {DATA_ROOT}/sample_submission.csv',
    f'train_image_dir: {DATA_ROOT}/train',
    f'test_image_dir: {DATA_ROOT}/test',
    'models_dir: /kaggle/working/models',
    'outputs_dir: /kaggle/working/outputs',
    'reports_dir: /kaggle/working/reports',
])
open('/kaggle/working/paths.yaml', 'w').write(yaml_text)
os.environ['SNUAI_PATHS_CONFIG'] = '/kaggle/working/paths.yaml'

# split.csv는 코드 번들에 있으므로 outputs_dir로 복사 (--fold val 경로가 참조)
os.makedirs('/kaggle/working/outputs', exist_ok=True)
shutil.copy(f'{CODE}/outputs/split.csv', '/kaggle/working/outputs/split.csv')

MODEL = glob.glob('/kaggle/input/*/qwen25vl3b_merged')[0]
print('DATA_ROOT:', DATA_ROOT, '| model:', MODEL)

In [ ]:
# (선택) val 리허설 — 제출 전 EM 확인. TTA/style은 학습과 일치시킬 것.
!cd $CODE && python -m src.infer.predict --model $MODEL --split train --fold val \
    --style mid --tta 4 --batch 16 --out /kaggle/working/raw_val.jsonl
!cd $CODE && python -m src.infer.aggregate --raw /kaggle/working/raw_val.jsonl \
    --out /kaggle/working/pred_val.csv
!cd $CODE && python -m src.eval.em --pred /kaggle/working/pred_val.csv --fold val

In [ ]:
# test 추론 → 집계 → 검증된 제출 CSV
!cd $CODE && python -m src.infer.predict --model $MODEL --split test \
    --style mid --tta 4 --batch 16 --out /kaggle/working/raw_test.jsonl
!cd $CODE && python -m src.infer.aggregate --raw /kaggle/working/raw_test.jsonl \
    --submission /kaggle/working/submission.csv

In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
print(sub.shape); sub.head()  # /kaggle/working/submission.csv 다운로드 → Kaggle 제출